In [1]:
from pyspark.sql import SparkSession
spark = (
  SparkSession
  .builder
  .appName('Sport_Predict')
  .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 20:20:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
sport_sdf = (
    spark
    .read
    .option('header', 'true')
    .csv("data/country_sports.csv")
)
sport_sdf.show()

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import DecisionTreeRegressor
sport_sdf = (
    spark
    .read
    .option('header', 'true')
    .schema('Country STRING, Continent STRING, Population INT, Sport STRING')
    .csv("assets/country_sports.csv")
)
va = VectorAssembler(
  inputCols=['Population'],
  outputCol='features'
)
va.transform(sport_sdf).select('Country','features','Sport').show()

In [ ]:
si = StringIndexer(
  inputCols=['Continent'],
  outputCols=['Continent_Index'],
  handleInvalid="skip"
)
fitted_si = si.fit(sport_sdf)
sport_ci_sdf = fitted_si.transform(sport_sdf)
sport_ci_sdf.select('Country','Continent','Continent_Index').show()

In [ ]:
from pyspark.ml.linalg import DenseVector
from pyspark.ml.functions import vector_to_array, array_to_vector
ohe = OneHotEncoder(
  inputCols=['Continent_Index'],
  outputCols=['Continent_OHE'],
)
fitted_ohe = ohe.fit(sport_ci_sdf)
sport_ohe_sdf = fitted_ohe.transform(sport_ci_sdf)
sport_dense_sdf = (
  sport_ohe_sdf
  .withColumn(
    "Continent_Dense",
    array_to_vector(
      vector_to_array("Continent_OHE")
    )
  )
)
(
  sport_dense_sdf
  .select('Country','Continent_Index','Continent_OHE','Continent_Dense')
).show()